# Improved House Price Model

This notebook builds the next submission after the `0.99172` score. The key extra improvement is training the final model on rows where `Lot_Depth` is known, because `house_test.csv` has no missing `Lot_Depth`. That keeps the model from learning from guessed lot sizes.

In [1]:
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import HuberRegressor, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_csv('house_train.csv')
test_df = pd.read_csv('house_test.csv')

complete_train_df = train_df.dropna(subset=['Lot_Depth']).copy()

print(f'Train shape: {train_df.shape}')
print(f'Complete Lot_Depth train shape: {complete_train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f"Missing Lot_Depth in test: {test_df['Lot_Depth'].isna().sum()}")
display(train_df.head())
display(test_df.head())

Train shape: (2000, 10)
Complete Lot_Depth train shape: (1721, 10)
Test shape: (1000, 9)
Missing Lot_Depth in test: 0


,ID,Location,Paint_Color,Lot_Width,Lot_Depth,Year_Sold,Year_Built,Living_Area,Num_Rooms,Price
0,1,Suburb,Green,134,NaN,2020,1990,813,6,1548800.0
1,2,Downtown,Green,76,93.0,2025,2019,3093,6,1974800.0
2,3,Suburb,Yellow,91,83.0,2019,2003,2845,3,2466950.0
3,4,Suburb,Green,26,21.0,2022,1995,1837,3,968700.0
4,5,Rural,Red,138,85.0,2017,1980,4928,8,2570700.0


,ID,Location,Paint_Color,Lot_Width,Lot_Depth,Year_Sold,Year_Built,Living_Area,Num_Rooms
0,2001,Rural,Green,106,131,2023,1980,1076,7
1,2002,Rural,Green,98,23,2016,1984,4828,2
2,2003,Downtown,Green,42,45,2020,2014,1405,5
3,2004,Rural,Red,100,36,2018,1991,4853,4
4,2005,Rural,Yellow,67,111,2021,2021,3512,6


In [3]:
print('Missing values in train:')
display(train_df.isna().sum())

print('Missing values in test:')
display(test_df.isna().sum())

Missing values in train:


ID               0
Location         0
Paint_Color      0
Lot_Width        0
Lot_Depth      279
Year_Sold        0
Year_Built       0
Living_Area      0
Num_Rooms        0
Price            0
dtype: int64

Missing values in test:


ID             0
Location       0
Paint_Color    0
Lot_Width      0
Lot_Depth      0
Year_Sold      0
Year_Built     0
Living_Area    0
Num_Rooms      0
dtype: int64

## Feature Engineering

The strongest signal in this synthetic-looking dataset comes from lot size, living area, and room-count interactions. The model also uses category interactions so each location/color can have a different slope.

In [4]:
def fit_feature_stats(df):
    return {
        'lot_depth_by_location': df.groupby('Location')['Lot_Depth'].median(),
        'lot_depth_global': df['Lot_Depth'].median(),
    }


def engineer_features(df, stats, include_target=False):
    data = df.copy()
    y = None

    if include_target and 'Price' in data.columns:
        y = data.pop('Price')

    data['Lot_Depth'] = data['Lot_Depth'].fillna(
        data['Location'].map(stats['lot_depth_by_location'])
    )
    data['Lot_Depth'] = data['Lot_Depth'].fillna(stats['lot_depth_global'])

    data['House_Age'] = (data['Year_Sold'] - data['Year_Built']).clip(lower=0)
    data['Lot_Area'] = data['Lot_Width'] * data['Lot_Depth']
    data['Living_Area_Per_Room'] = data['Living_Area'] / data['Num_Rooms'].replace(0, np.nan)
    data['Lot_Area_Per_Room'] = data['Lot_Area'] / data['Num_Rooms'].replace(0, np.nan)
    data['Lot_Living_Per_Room'] = data['Lot_Area'] * data['Living_Area'] / data['Num_Rooms'].replace(0, np.nan)
    data['Lot_Living_Per_Room_SqRooms'] = (
        data['Lot_Area'] * data['Living_Area'] / (data['Num_Rooms'].replace(0, np.nan) ** 2)
    )
    data['Living_Area_Sq_Per_Room'] = data['Living_Area'] ** 2 / data['Num_Rooms'].replace(0, np.nan)
    data['Living_To_Lot_Ratio'] = data['Living_Area'] / data['Lot_Area'].replace(0, np.nan)
    data['Lot_Width_Depth_Ratio'] = data['Lot_Width'] / data['Lot_Depth'].replace(0, np.nan)
    data['Width_Living_Per_Room'] = data['Lot_Width'] * data['Living_Area'] / data['Num_Rooms'].replace(0, np.nan)
    data['Depth_Living_Per_Room'] = data['Lot_Depth'] * data['Living_Area'] / data['Num_Rooms'].replace(0, np.nan)
    data['Age_Living_Per_Room'] = data['House_Age'] * data['Living_Area'] / data['Num_Rooms'].replace(0, np.nan)
    data['Age_Lot_Area'] = data['House_Age'] * data['Lot_Area']
    data['Age_Lot_Living_Per_Room'] = (
        data['House_Age'] * data['Lot_Area'] * data['Living_Area'] / data['Num_Rooms'].replace(0, np.nan)
    )

    numeric_for_transforms = [
        'Lot_Width',
        'Lot_Depth',
        'Living_Area',
        'Num_Rooms',
        'House_Age',
        'Lot_Area',
        'Living_Area_Per_Room',
        'Lot_Area_Per_Room',
        'Lot_Living_Per_Room',
    ]
    for column in numeric_for_transforms:
        non_negative = data[column].clip(lower=0)
        data[f'Log_{column}'] = np.log1p(non_negative)
        data[f'{column}_Sq'] = data[column] ** 2
        data[f'{column}_Sqrt'] = np.sqrt(non_negative)

    data = data.drop(columns=['ID'], errors='ignore')
    data = pd.get_dummies(
        data,
        columns=['Location', 'Paint_Color'],
        drop_first=False,
        dtype=int,
    )

    interaction_features = [
        'Living_Area',
        'Lot_Area',
        'Living_Area_Per_Room',
        'Lot_Living_Per_Room',
        'House_Age',
        'Num_Rooms',
    ]
    category_columns = [
        column for column in data.columns
        if column.startswith('Location_') or column.startswith('Paint_Color_')
    ]
    for category in category_columns:
        for feature in interaction_features:
            data[f'{category}_x_{feature}'] = data[category] * data[feature]

    data = data.replace([np.inf, -np.inf], np.nan)
    return data, y

In [5]:
all_stats = fit_feature_stats(train_df)
complete_stats = fit_feature_stats(complete_train_df)

X_all, y_all = engineer_features(train_df, all_stats, include_target=True)
X_complete, y_complete = engineer_features(complete_train_df, complete_stats, include_target=True)

all_feature_columns = X_all.columns
complete_feature_columns = X_complete.columns

all_feature_medians = X_all.median(numeric_only=True)
complete_feature_medians = X_complete.median(numeric_only=True)

X_all = X_all.fillna(all_feature_medians)
X_complete = X_complete.fillna(complete_feature_medians)

print(f'All-row feature matrix: {X_all.shape}')
print(f'Complete-row feature matrix: {X_complete.shape}')
display(X_complete.head())

All-row feature matrix: (2000, 96)
Complete-row feature matrix: (1721, 96)


,Lot_Width,Lot_Depth,Year_Sold,Year_Built,Living_Area,Num_Rooms,House_Age,Lot_Area,Living_Area_Per_Room,Lot_Area_Per_Room,...,Paint_Color_Red_x_Living_Area_Per_Room,Paint_Color_Red_x_Lot_Living_Per_Room,Paint_Color_Red_x_House_Age,Paint_Color_Red_x_Num_Rooms,Paint_Color_Yellow_x_Living_Area,Paint_Color_Yellow_x_Lot_Area,Paint_Color_Yellow_x_Living_Area_Per_Room,Paint_Color_Yellow_x_Lot_Living_Per_Room,Paint_Color_Yellow_x_House_Age,Paint_Color_Yellow_x_Num_Rooms
1,76,93.0,2025,2019,3093,6,6,7068.0,515.500000,1178.000000,...,0.0,0.0,0,0,0,0.0,0.000000,0.000000e+00,0,0
2,91,83.0,2019,2003,2845,3,16,7553.0,948.333333,2517.666667,...,0.0,0.0,0,0,2845,7553.0,948.333333,7.162762e+06,16,3
3,26,21.0,2022,1995,1837,3,27,546.0,612.333333,182.000000,...,0.0,0.0,0,0,0,0.0,0.000000,0.000000e+00,0,0
4,138,85.0,2017,1980,4928,8,37,11730.0,616.000000,1466.250000,...,616.0,7225680.0,37,8,0,0.0,0.000000,0.000000e+00,0,0
5,133,50.0,2021,2000,2259,7,21,6650.0,322.714286,950.000000,...,0.0,0.0,0,0,0,0.0,0.000000,0.000000e+00,0,0


## Validation

This compares the previous all-row strategy with the complete-row strategy that better matches the test file.

In [6]:
def make_log_model(regressor):
    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )


def clipped_predict(model, X):
    return np.clip(model.predict(X), a_min=0, a_max=None)


def blended_predictions(ridge_model, huber_model, X, ridge_weight=0.35):
    ridge_pred = clipped_predict(ridge_model, X)
    huber_pred = clipped_predict(huber_model, X)
    blend_log = ridge_weight * np.log1p(ridge_pred) + (1 - ridge_weight) * np.log1p(huber_pred)
    return np.expm1(blend_log)


def regression_metrics(y_true, y_pred):
    y_pred = np.clip(y_pred, a_min=0, a_max=None)
    return {
        'r2_price_scale': r2_score(y_true, y_pred),
        'r2_log_scale': r2_score(np.log1p(y_true), np.log1p(y_pred)),
        'rmsle': mean_squared_log_error(y_true, y_pred) ** 0.5,
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
    }

In [7]:
def validate_blend(X, y, ridge_weight=0.35):
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    linear_model = make_log_model(LinearRegression())
    ridge_model = make_log_model(make_pipeline(StandardScaler(), Ridge(alpha=10)))
    huber_model = make_log_model(
        make_pipeline(
            StandardScaler(),
            HuberRegressor(epsilon=1.35, alpha=0.001, max_iter=3000),
        )
    )

    linear_model.fit(X_train, y_train)
    ridge_model.fit(X_train, y_train)
    huber_model.fit(X_train, y_train)

    return pd.DataFrame(
        {
            'linear_log': regression_metrics(y_valid, clipped_predict(linear_model, X_valid)),
            'ridge_log': regression_metrics(y_valid, clipped_predict(ridge_model, X_valid)),
            'huber_log': regression_metrics(y_valid, clipped_predict(huber_model, X_valid)),
            'ridge_huber_blend': regression_metrics(
                y_valid,
                blended_predictions(ridge_model, huber_model, X_valid, ridge_weight=ridge_weight),
            ),
        }
    ).T

all_row_metrics = validate_blend(X_all, y_all, ridge_weight=0.40)
complete_row_metrics = validate_blend(X_complete, y_complete, ridge_weight=0.35)

print('All-row validation')
display(all_row_metrics)

print('Complete-row validation')
display(complete_row_metrics)

All-row validation


,r2_price_scale,r2_log_scale,rmsle,mae,rmse
linear_log,0.407746,0.743982,0.231254,259942.576048,1.035701e+06
ridge_log,0.412349,0.747105,0.229839,271786.671475,1.031669e+06
huber_log,0.406238,0.759309,0.224225,180192.614926,1.037019e+06
ridge_huber_blend,0.412677,0.760269,0.223777,210596.069042,1.031381e+06


Complete-row validation


,r2_price_scale,r2_log_scale,rmsle,mae,rmse
linear_log,0.310848,0.710355,0.285039,363475.929704,1.570745e+06
ridge_log,0.327564,0.709349,0.285534,355781.823656,1.551578e+06
huber_log,0.296270,0.707976,0.286207,256303.164855,1.587271e+06
ridge_huber_blend,0.309123,0.712587,0.283939,284496.062651,1.572709e+06


In [8]:
def cross_validated_blend_score(X, y, ridge_weight=0.35):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    ridge_model = make_log_model(make_pipeline(StandardScaler(), Ridge(alpha=10)))
    huber_model = make_log_model(
        make_pipeline(
            StandardScaler(),
            HuberRegressor(epsilon=1.35, alpha=0.001, max_iter=3000),
        )
    )
    ridge_pred = np.clip(cross_val_predict(ridge_model, X, y, cv=cv), a_min=0, a_max=None)
    huber_pred = np.clip(cross_val_predict(huber_model, X, y, cv=cv), a_min=0, a_max=None)
    blend_pred = np.expm1(
        ridge_weight * np.log1p(ridge_pred)
        + (1 - ridge_weight) * np.log1p(huber_pred)
    )
    return pd.Series(regression_metrics(y, blend_pred))

cv_summary = pd.DataFrame(
    {
        'all_rows_blend_w_0_40': cross_validated_blend_score(X_all, y_all, ridge_weight=0.40),
        'complete_rows_blend_w_0_35': cross_validated_blend_score(X_complete, y_complete, ridge_weight=0.35),
    }
).T
cv_summary

,r2_price_scale,r2_log_scale,rmsle,mae,rmse
all_rows_blend_w_0_40,0.311830,0.717118,0.270170,297129.730910,1.477645e+06
complete_rows_blend_w_0_35,0.338654,0.768244,0.243964,233101.104335,1.453159e+06


## Final Submission

The final model below is trained on complete `Lot_Depth` rows only, because the test set has complete `Lot_Depth`.

In [9]:
final_ridge_model = make_log_model(make_pipeline(StandardScaler(), Ridge(alpha=10)))
final_huber_model = make_log_model(
    make_pipeline(
        StandardScaler(),
        HuberRegressor(epsilon=1.35, alpha=0.001, max_iter=3000),
    )
)

final_ridge_model.fit(X_complete, y_complete)
final_huber_model.fit(X_complete, y_complete)

X_house_test, y_house_test = engineer_features(
    test_df,
    complete_stats,
    include_target=('Price' in test_df.columns),
)
X_house_test = X_house_test.reindex(columns=complete_feature_columns, fill_value=0)
X_house_test = X_house_test.fillna(complete_feature_medians)

house_test_price_pred = blended_predictions(
    final_ridge_model,
    final_huber_model,
    X_house_test,
    ridge_weight=0.35,
)
house_test_price_pred = np.clip(house_test_price_pred, a_min=0, a_max=None)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Price': house_test_price_pred,
})
submission.to_csv('house_test_predictions.csv', index=False)
submission.to_csv('submission4.csv', index=False)

submission.head()

,ID,Price
0,2001,2.285840e+06
1,2002,3.450221e+06
2,2003,9.110004e+05
3,2004,2.125631e+06
4,2005,2.095044e+06


In [10]:
if y_house_test is not None:
    print('house_test.csv metrics:')
    display(pd.Series(regression_metrics(y_house_test, house_test_price_pred)))
else:
    print('house_test.csv has no Price column, so true test R2 cannot be calculated locally.')
    print('Use submission4.csv as the next submission candidate.')
    print(f"Complete-row CV log R2: {cv_summary.loc['complete_rows_blend_w_0_35', 'r2_log_scale']:.4f}")
    print(f"Complete-row CV RMSLE: {cv_summary.loc['complete_rows_blend_w_0_35', 'rmsle']:.4f}")
    print('Saved predictions to house_test_predictions.csv and submission4.csv')

house_test.csv has no Price column, so true test R2 cannot be calculated locally.
Use submission4.csv as the next submission candidate.
Complete-row CV log R2: 0.7682
Complete-row CV RMSLE: 0.2440
Saved predictions to house_test_predictions.csv and submission4.csv


## Label-Corruption Cleanup Candidates

The robust model reveals a small group of training rows whose prices are about `5x` too high. These candidate submissions train on cleaner targets and are intended to improve beyond the previous `0.99477` score.

In [11]:
def make_clean_huber_model():
    return make_log_model(
        make_pipeline(
            StandardScaler(),
            HuberRegressor(epsilon=1.35, alpha=0.001, max_iter=3000),
        )
    )


def make_clean_ridge_model():
    return make_log_model(make_pipeline(StandardScaler(), Ridge(alpha=10)))


cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof_huber_pred = np.clip(
    cross_val_predict(make_clean_huber_model(), X_complete, y_complete, cv=cv),
    a_min=0,
    a_max=None,
)
log_residual = pd.Series(
    np.abs(np.log1p(y_complete) - np.log1p(oof_huber_pred)),
    index=complete_train_df.index,
)
pred_ratio = pd.Series(y_complete.values / oof_huber_pred, index=complete_train_df.index)
factor5_mask = pred_ratio.between(4.7, 5.3)

correct_factor5_df = complete_train_df.copy()
correct_factor5_df.loc[factor5_mask, 'Price'] = correct_factor5_df.loc[factor5_mask, 'Price'] / 5
trim975_df = complete_train_df.loc[log_residual <= log_residual.quantile(0.975)].copy()
trim95_df = complete_train_df.loc[log_residual <= log_residual.quantile(0.95)].copy()
trim90_df = complete_train_df.loc[log_residual <= log_residual.quantile(0.90)].copy()

print(f'Factor-5 labels detected: {factor5_mask.sum()}')
print(f'Trim 97.5% rows: {len(trim975_df)}')
print(f'Trim 95% rows: {len(trim95_df)}')
print(f'Trim 90% rows: {len(trim90_df)}')


def make_submission_from_clean_train(clean_train_df, filename):
    clean_stats = fit_feature_stats(clean_train_df)
    X_clean, y_clean = engineer_features(clean_train_df, clean_stats, include_target=True)
    clean_columns = X_clean.columns
    clean_medians = X_clean.median(numeric_only=True)
    X_clean = X_clean.fillna(clean_medians)

    X_clean_test, _ = engineer_features(test_df, clean_stats, include_target=False)
    X_clean_test = X_clean_test.reindex(columns=clean_columns, fill_value=0)
    X_clean_test = X_clean_test.fillna(clean_medians)

    model = make_clean_huber_model()
    model.fit(X_clean, y_clean)
    clean_pred = np.clip(model.predict(X_clean_test), a_min=0, a_max=None)

    clean_submission = pd.DataFrame({'ID': test_df['ID'], 'Price': clean_pred})
    clean_submission.to_csv(filename, index=False)
    return clean_pred


pred6 = make_submission_from_clean_train(correct_factor5_df, 'submission6_correct_factor5.csv')
pred7 = make_submission_from_clean_train(trim95_df, 'submission7_trim95.csv')
pred8 = make_submission_from_clean_train(trim90_df, 'submission8_trim90.csv')
pred9 = make_submission_from_clean_train(trim975_df, 'submission9_trim975.csv')

pred10 = np.expm1(0.5 * np.log1p(pred7) + 0.5 * np.log1p(pred8))
pd.DataFrame({'ID': test_df['ID'], 'Price': pred10}).to_csv(
    'submission10_trim95_trim90_blend.csv',
    index=False,
)

pd.DataFrame({'ID': test_df['ID'], 'Price': pred8}).to_csv('house_test_predictions.csv', index=False)

pd.DataFrame(
    {
        'candidate': [
            'submission6_correct_factor5.csv',
            'submission7_trim95.csv',
            'submission8_trim90.csv',
            'submission9_trim975.csv',
            'submission10_trim95_trim90_blend.csv',
        ],
        'training_rows': [len(correct_factor5_df), len(trim95_df), len(trim90_df), len(trim975_df), 'blend'],
        'note': [
            'correct factor-5 labels',
            'drop noisiest 5%',
            'drop noisiest 10%',
            'drop noisiest 2.5%',
            'blend trim95 and trim90 predictions',
        ],
    }
)

Factor-5 labels detected: 40
Trim 97.5% rows: 1678
Trim 95% rows: 1635
Trim 90% rows: 1549


,candidate,training_rows,note
0,submission6_correct_factor5.csv,1721,correct factor-5 labels
1,submission7_trim95.csv,1635,drop noisiest 5%
2,submission8_trim90.csv,1549,drop noisiest 10%
3,submission9_trim975.csv,1678,drop noisiest 2.5%
4,submission10_trim95_trim90_blend.csv,blend,blend trim95 and trim90 predictions
